# EDA — NeuroLearn Analytics (Versão 1)

## Objetivo

Explorar o dataset sintético alinhado a seis domínios: atenção/FE, memória de trabalho, linguagem oral, leitura, escrita e aritmética — e a associação com `indicador_vulnerabilidade_aprendizagem`.

## Disclaimer

> Este notebook utiliza exclusivamente dados sintéticos e possui finalidade acadêmica e demonstrativa. As relações observadas são resultado da lógica de simulação utilizada no projeto e não devem ser interpretadas como evidência clínica, diagnóstica ou epidemiológica.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import (
    CONTEXT_FEATURES,
    DOMAIN_SCORE_COLUMNS,
    MODEL_FEATURES,
    N_SAMPLES,
    RANDOM_STATE,
    TARGET_COLUMN,
    TRANSVERSAL_FEATURES,
)
from src.data_generator import generate_synthetic_dataset
from src.eda import (
    SCATTER_PAIRS,
    count_iqr_outliers,
    find_high_correlations,
    generate_eda_figures,
    get_top_risk_correlations,
    plot_context_overview,
    plot_correlation_matrix,
    plot_domain_score_distributions,
    plot_domain_scores_by_risk,
    plot_risk_distribution,
    plot_scatter_relation,
    plot_top_risk_correlations,
    summarize_dataset,
)
from src.preprocessing import add_domain_scores

FIGURES_DIR = ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 40)

In [ ]:
df_raw = generate_synthetic_dataset(n_samples=N_SAMPLES, random_state=RANDOM_STATE)
df = add_domain_scores(df_raw)
print("Shape:", df.shape)
print("Target:", TARGET_COLUMN)
print("Scores:", DOMAIN_SCORE_COLUMNS)
df.head()

In [ ]:
summary = summarize_dataset(df)
print("Duplicados:", summary["n_duplicates"])
print("NaN total:", summary["n_missing_total"])
print(summary["target_counts"])
print(summary["target_proportions"].round(4))
df.info()

In [ ]:
df.describe().T.round(2)

In [ ]:
fig, ax = plot_risk_distribution(df, FIGURES_DIR / "risk_distribution.png")
plt.show()
plt.close(fig)

In [ ]:
print(df[DOMAIN_SCORE_COLUMNS].agg(["mean", "median", "std", "min", "max"]).round(2).T)
fig = plot_domain_score_distributions(df, output_path=FIGURES_DIR / "domain_scores_distribution.png")
plt.show()
plt.close(fig)

In [ ]:
fig, ax = plot_domain_scores_by_risk(df, FIGURES_DIR / "domain_scores_by_risk.png")
plt.show()
plt.close(fig)
df.groupby(TARGET_COLUMN)[DOMAIN_SCORE_COLUMNS].mean().round(2)

In [ ]:
fig, ax = plot_correlation_matrix(df, output_path=FIGURES_DIR / "correlation_matrix.png")
plt.show()
plt.close(fig)
print("Correlação entre os 6 scores:")
display(df[DOMAIN_SCORE_COLUMNS].corr().round(3))

In [ ]:
top_corr = get_top_risk_correlations(df, top_n=10)
display(top_corr)
fig, ax = plot_top_risk_correlations(df, top_n=10, output_path=FIGURES_DIR / "top_risk_correlations.png")
plt.show()
plt.close(fig)

In [ ]:
for x, y in SCATTER_PAIRS:
    fig, ax = plot_scatter_relation(df, x, y, FIGURES_DIR / f"scatter_{x}_vs_{y}.png")
    plt.show()
    plt.close(fig)

In [ ]:
print(df[TRANSVERSAL_FEATURES + CONTEXT_FEATURES].describe().T.round(2))
fig = plot_context_overview(df, FIGURES_DIR / "context_distributions.png")
plt.show()
plt.close(fig)

In [ ]:
high_corr = find_high_correlations(df, columns=MODEL_FEATURES, threshold=0.90)
if high_corr.empty:
    print("Nenhum par de features individuais com |r| > 0.90.")
else:
    display(high_corr)

display(count_iqr_outliers(df, DOMAIN_SCORE_COLUMNS + ["taxa_erros", "fadiga", "tempo_resposta"]))

In [ ]:
paths = generate_eda_figures(df, FIGURES_DIR)
for name, path in paths.items():
    print(f"{path.name}: {path.stat().st_size / 1024:.1f} KB")

## Limitações e conclusões exploratórias

- Dataset 100% sintético; padrões refletem a lógica de geração da Versão 1.
- Correlação ≠ causalidade; scores não são escalas clínicas.
- Classe positiva do indicador é minoritária, sem desbalanceamento extremo.
- Os seis scores apresentam correlações moderadas, coerentes com fatores latentes partilhados (ex.: fonologia ↔ leitura/escrita).
- Cognição social está fora do núcleo desta versão.
- Não interpretar resultados como evidência clínica ou epidemiológica.